# CP1 Week 10 -- Functions: Decomposition & Reuse

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Apply the DRY principle (Don't Repeat Yourself)
2. Write functions with default parameters
3. Add docstrings to all pipeline functions
4. Improve code organization through decomposition

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: DRY -- Don't Repeat Yourself

In [ ]:
# BAD: same logic repeated
def process_temps(data):
    total = 0
    for v in data:
        total += v
    return total / len(data)

def process_pressures(data):
    total = 0
    for v in data:
        total += v
    return total / len(data)

# GOOD: one reusable function
def mean(data):
    """Calculate mean of any numeric list."""
    if not data:
        return 0
    return sum(data) / len(data)

temps = [20, 25, 30]
pressures = [101, 102, 100]
print(f"Mean temp: {mean(temps)}")
print(f"Mean pressure: {mean(pressures)}")

**Expected Output:**
```
Mean temp: 25.0
Mean pressure: 101.0
```

### Example 2 -- Finding duplication

In [ ]:
# Spot the duplication in these two functions:

def report_temps(data):
    if not data:
        print("No data")
        return
    total = sum(data)
    count = len(data)
    avg = total / count
    print(f"Temperature: avg={avg:.2f}, min={min(data)}, max={max(data)}, n={count}")

def report_rpms(data):
    if not data:
        print("No data")
        return
    total = sum(data)
    count = len(data)
    avg = total / count
    print(f"RPM: avg={avg:.2f}, min={min(data)}, max={max(data)}, n={count}")

# The ONLY difference is the label! Let's DRY it:
def report_metric(data, label="Value"):
    """Report stats for any metric -- DRY version."""
    if not data:
        print(f"{label}: No data")
        return
    avg = sum(data) / len(data)
    print(f"{label}: avg={avg:.2f}, min={min(data)}, max={max(data)}, n={len(data)}")

report_metric([20, 25, 30], "Temperature")
report_metric([1500, 1520, 1480], "RPM")
report_metric([12, 13, 11], "Vibration")

**Expected Output:**
```
Temperature: avg=25.00, min=20, max=30, n=3
RPM: avg=1500.00, min=1480, max=1520, n=3
Vibration: avg=12.00, min=11, max=13, n=3
```

One function replaced two, and it handles ANY metric!

### Try It Yourself #1

In [ ]:
# TODO: Find the duplication below and refactor into one function.

def validate_temp(value):
    if value is None:
        return False
    if not isinstance(value, (int, float)):
        return False
    if value < 0 or value > 150:
        return False
    return True

def validate_rpm(value):
    if value is None:
        return False
    if not isinstance(value, (int, float)):
        return False
    if value < 0 or value > 5000:
        return False
    return True

# Refactored version:
# def validate_value(value, min_val=0, max_val=100):
#     ...

# Test it:
# print(validate_value(25, 0, 150))
# print(validate_value(1500, 0, 5000))

---
## Part 2: Default Parameters

In [ ]:
def clean_column(data, column="value", min_val=0, max_val=100, drop_missing=True):
    """Clean a specific column from data rows.

    Args:
        data: list of dicts
        column: which key to clean (default: "value")
        min_val: minimum valid value (default: 0)
        max_val: maximum valid value (default: 100)
        drop_missing: skip rows with missing values (default: True)

    Returns:
        list of dicts with valid values
    """
    result = []
    for row in data:
        val = row.get(column)
        if val is None and drop_missing:
            continue
        try:
            num = float(val)
        except (ValueError, TypeError):
            continue
        if min_val <= num <= max_val:
            result.append({**row, column: num})
    return result

data = [{"value": "25"}, {"value": ""}, {"value": "50"}, {"value": "200"}]
print("Default:", clean_column(data))
print("Custom:", clean_column(data, min_val=30, max_val=100))

---
## Part 3: Docstrings

Every function should have a docstring explaining what it does, what it takes,
and what it returns.

In [ ]:
def analyze_pipeline(data, config):
    """Run the full analysis pipeline.

    This function orchestrates data analysis:
    1. Extract numeric values
    2. Compute summary statistics
    3. Classify each value
    4. Return structured results

    Args:
        data (list[dict]): Cleaned data rows with 'value' key.
        config (dict): Configuration with optional 'threshold'.

    Returns:
        dict: Results with 'stats', 'labels', 'summary' keys.

    Example:
        >>> results = analyze_pipeline([{"value": 10}], {"threshold": 5})
        >>> results["stats"]["mean"]
        10.0
    """
    values = [row["value"] for row in data]
    if not values:
        return {"stats": {}, "labels": [], "summary": "No data"}

    m = sum(values) / len(values)
    threshold = config.get("threshold", m * 1.5)
    labels = ["high" if v > threshold else "normal" for v in values]

    return {
        "stats": {"count": len(values), "mean": round(m, 2)},
        "labels": labels,
        "summary": str(len(values)) + " values analyzed, " + str(labels.count("high")) + " high"
    }

result = analyze_pipeline([{"value": 10}, {"value": 50}, {"value": 20}], {"threshold": 30})
print(result["summary"])

# View docstring
help(analyze_pipeline)

### Example -- Three ways to call a function with defaults

In [ ]:
def format_value(value, decimals=2, prefix="", suffix="", width=0):
    """Format a numeric value for display."""
    formatted = f"{value:.{decimals}f}"
    result = prefix + formatted + suffix
    if width > 0:
        result = result.rjust(width)
    return result

# Way 1: All defaults
print(format_value(3.14159))

# Way 2: Some overrides
print(format_value(72.5, decimals=1, suffix=" C"))

# Way 3: Named arguments in any order
print(format_value(1500, decimals=0, prefix="RPM: ", width=15))

**Expected Output:**
```
3.14
72.5 C
       RPM: 1500
```

### Common Mistakes with Functions

| Mistake | Example | Fix |
|---------|---------|-----|
| Mutable default arg | `def f(data=[]):` | `def f(data=None):` then `data = data or []` |
| No return statement | `def add(a,b): a+b` | `def add(a,b): return a+b` |
| Modifying input | `data.sort(); return data` | `return sorted(data)` |
| Too many params | `def f(a,b,c,d,e,f,g):` | Group into a config dict |

---
## Key Takeaways -- Week 10

1. **DRY**: Extract repeated logic into shared functions
2. **Default parameters** make functions flexible yet easy to call
3. **Docstrings** document what, why, args, and returns
4. **Decomposition** means breaking big functions into small, focused helpers
5. **One function, one job** -- each function should do exactly one thing

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What does DRY stand for?
# R2: What is a default parameter?
# R3: What should a docstring include?
# R4: What is function decomposition?

### Practice (P1-P5)

In [ ]:
# P1: Find 3 places in your project with repeated logic. Refactor.


In [ ]:
# P2: Add docstrings to ALL your pipeline functions.


In [ ]:
# P3: Write 5 reusable helper functions: mean, median, std, is_numeric, safe_float


In [ ]:
# P4: Create a function with 4+ default parameters. Show 3 call styles.


In [ ]:
# P5: Write a "utility module" as a dict of functions.


### Challenge (C1-C2)

In [ ]:
# C1: Write a function that generates a formatted report for any dict of stats.


In [ ]:
# C2: Write a simple function decorator that logs function calls.


### Mini-Project

In [ ]:
# M1: Pipeline Cleanup
# Take your ENTIRE pipeline code and:
# 1. Remove ALL code duplication
# 2. Add docstrings to EVERY function
# 3. Give every function default parameters where sensible
# 4. Create at least 3 new helper functions
# 5. Add a print_report() function that formats results nicely


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)